# Object-oriented scientific programming with C++

Matthias Möller, Jonas Thies

Lecture 3 - 25 November 2026

# Learning objectives of this lecture

- Overloading of functions and operators
- Class inheritance
- Runtime polymorphism: _virtual methods, override, and abstract classes_
- Template meta-programming and specialization
- Automatic type decutaion: _auto_

# Block A: Overloading

We want to implement a C++ class `Interval` that represents a _closed_ interval \\(I=[\text{low}, \text{high}]\subset \mathbb{R}\\) and allows to check if 

- a given point \\(x\in\mathbb{R}\\) is inside or outside
- a given interval \\(J\\) is fully contained in \\(I\\) or not

While designing this class, we already keep in mind that we will afterwards extend it towards _open_ intervals.

# Step A.1: Creating an initial class `Interval`

__Prompt:__ _???_

We want to be able to run the following `main` function:

```c++
int main(){
    Interval A{0, 5}, B{2,3};
    std::cout << "A contains 3? " << A.contains(3.0) << std::endl;
    std::cout << "B contains 4? " << B.contains(4.0) << std::endl;
    std::cout << "A contains B? " << A.contains(B) << std::endl;
    return 0;
}
```

__Prompt:__ _Make a C++ `struct` called `Interval` that stores a lower bound `low` and an upper bound `high` as
doubles, with a constructor taking both. Add a method `contains(double x)` that returns whether
x lies inside the closed interval. Then add another method `contains(const Interval& other)`
that returns whether the whole other interval lies inside this one. Also add a function that returns
the width of the interval. In `main`, build an interval \\([0, 10]\\), test a point inside, a point outside,
and a smaller interval inside._

```c++
#include <iostream>
struct Interval {
    double high, low;

    Interval(double low, double high): low(low), high(high) {}

    bool contains(double x) const {
        return (low <= x) && (x <= high);
    }
    bool contains(const Interval& other) const {
        return (low <= other.low) && (other.high <= high);
    }
    
    double width() const {
        return high - low;
    }
};

int main(){
    Interval A{0, 5}, B{2,3};
    std::cout << "A contains 3? " << A.contains(3.0) << std::endl;
    std::cout << "B contains 4? " << B.contains(4.0) << std::endl;
    std::cout << "A contains B? " << A.contains(B) << std::endl;
    return 0;
}
```

# Why is this lecture block called 'Overloading'?

```c++
bool contains(double x)              const { ... }
bool contains(const Interval& other) const { ... }
```

__Overloading__ allows functions or operators with the _same name_ to behave differently depending on the number or types of their arguments.

The return type does not distinguish overloads: two functions cannot differ solely in their return type. Any idea why?

# Quiz: Who decides which `contains` function to call?

- __Stand up__ if you think that the __runtime system__ decides.
- __Stay seated__ if you think that the __compiler__ decides.

__Correct answer:__ The __compiler__ decides. This is why _overloading_ is called __compile-time polymorphism__.

# Quiz: Is this a valid overload?

```c++
int contains(double x) const {
    if (x < low) return -1;
    if (x > high) return 1;
    else return 0;
}
```

No, because the only difference is in the return type.

# Quiz: Is this a valid overload?

```c++
int contains(double y) const {
	return (low <= y) && (y <= high);
}
```

No, because the compiler does not consider the parameter's name but only its type.

# Quiz: Is this a valid overload?

```c++
bool contains(double x, double tol = 0.0) const {
    return (low <= x + tol) && (x - tol <= high);
}
```

No, because the default parameter can be omitted and then it is not clear which overload to use.

# Block B: Creating an _open_ `Interval` class

We now want to implement the same functionality for an _open_ interval \\(I=(\text{low}, \text{high})\subset \mathbb{R}\\).

__Think:__

- Which parts of the existing implementation remain the same, and which need to change?
- How would you reuse the shared functionality while adapting the behavior that differs?

__Prompt:__ _???_

__Prompt:__ _Create a new child class of `Interval` called `OpenInterval`. For now, simply add a constructor delegating to `Interval`'s constructor. Create a new `OpenInterval` and try a few relevant values on it._

```c++
class OpenInterval : public Interval {
	OpenInterval(double low, double high)
        : Interval(low,high) 
    {}
};

int main() {
    Interval A{0, 5};
    std::cout << "A contains 3? " << A.contains(3.0) << std::endl;
    std::cout << "B contains 4? " << B.contains(4.0) << std::endl;
    
    OpenInterval B{2, 4};
    std::cout << "width of B? " << B.width() << std::endl;
    std::cout << "B contains 4? " << B.contains(4.0) << std::endl;
    std::cout << "B contains A? " << B.contains(A) << std::endl;
    return 0;
}
```

# Inheritance

__Syntax:__
```c++
class DerivedClass : public BaseClass { ... };
```

`DerivedClass` __inherits__ data members and member functions from `BaseClass`.

With `public` inheritance:

- __Public members__ of `BaseClass` remain public in `DerivedClass`.

- __Protected members__ of `BaseClass` remain protected in `DerivedClass`.

- __Private members__ of `BaseClass` cannot be accessed directly by `DerivedClass`.